In [1]:
from scripts.config import cfg

checks = cfg.check_credentials()
for name, ok in checks:
    print(f"{'✓' if ok else '✗'} {name}")

✓ CHARTMETRIC_REFRESH_TOKEN
✓ LUMINATE_API_KEY
✓ LUMINATE_EMAIL
✓ LUMINATE_PASSWORD
✓ SOUNDCLOUD_CLIENT_ID


In [2]:
from scripts.config import cfg
from scripts.platforms.chartmetric import ChartmetricClient

cm = ChartmetricClient()
artist = cm.find_artist("Drake")
print(artist)

{'id': 3380, 'name': 'Drake', 'image_url': 'https://i.scdn.co/image/ab6761610000e5eb4293385d324db8558179afd9', 'isni': '000000012032246X', 'verified': True, 'sp_followers': 107304801, 'sp_monthly_listeners': 88579902, 'cm_artist_score': 98.78956754510065, 'band': False, 'gender': 'male', 'primary_genre_smart': 501121}


In [3]:
from scripts.platforms.chartmetric import ChartmetricClient

cm = ChartmetricClient()

# Find the artist

artist = cm.find_artist("Avello")

print(artist)

{'id': 3623359, 'name': 'AVELLO', 'image_url': 'https://i.scdn.co/image/ab6761610000e5eb1a61a6367dead8dac77f1911', 'isni': None, 'verified': False, 'sp_followers': 24374, 'sp_monthly_listeners': 1461543, 'cm_artist_score': 70.56697879855538, 'band': None, 'gender': None, 'primary_genre_smart': 501303}


In [6]:
raw = cm.get_artist_stats(3623359, "instagram")
print(type(raw))
print(raw)

<class 'dict'>
{'link': 'https://www.instagram.com/avello.music', 'followers': [{'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_diff': 2097, 'monthly_diff_percent': 2.8, 'value': 75425, 'timestp': '2026-02-01T00:00:00.000Z', 'diff': None}, {'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_diff': 2097, 'monthly_diff_percent': 2.8, 'value': 75575, 'timestp': '2026-02-02T00:00:00.000Z', 'diff': 150}, {'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_diff': 2097, 'monthly_diff_percent': 2.8, 'value': 75647, 'timestp': '2026-02-03T00:00:00.000Z', 'diff': 72}, {'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_diff': 2097, 'monthly_diff_percent': 2.8, 'value': 75710, 'timestp': '2026-02-04T00:00:00.000Z', 'diff': 63}, {'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_diff': 2097, 'monthly_diff_percent': 2.8, 'value': 75821, 'timestp': '2026-02-05T00:00:00.000Z', 'diff': 111}, {'weekly_diff': 476, 'weekly_diff_percent': 0.6221, 'monthly_

In [7]:
import pandas as pd
import time

cm_id = 3623359

def get_latest_stat(platform):
    time.sleep(0.3)
    try:
        data = cm.get_artist_stats(cm_id, platform)
        if isinstance(data, dict):
            # platforms like instagram return {link, followers: [...]}
            for key in ["followers", "listeners", "subscribers", "fans"]:
                if key in data and data[key]:
                    latest = data[key][-1]
                    return {
                        "value": latest.get("value"),
                        "weekly_diff": latest.get("weekly_diff"),
                        "monthly_diff": latest.get("monthly_diff"),
                        "weekly_diff_pct": latest.get("weekly_diff_percent"),
                    }
        elif isinstance(data, list) and data:
            latest = data[-1]
            return {
                "value": latest.get("value"),
                "weekly_diff": latest.get("weekly_diff"),
                "monthly_diff": latest.get("monthly_diff"),
                "weekly_diff_pct": latest.get("weekly_diff_percent"),
            }
    except:
        return {}
    return {}

platforms = ["instagram", "tiktok", "youtube_channel", "twitter", "soundcloud", "deezer"]

row = {
    "artist": "AVELLO",
    "cm_id": cm_id,
    "cm_artist_score": 70.57,
    "sp_followers": 24374,
    "sp_monthly_listeners": 1461543,
}

for platform in platforms:
    stats = get_latest_stat(platform)
    row[f"{platform}_followers"] = stats.get("value")
    row[f"{platform}_weekly_diff"] = stats.get("weekly_diff")
    row[f"{platform}_weekly_diff_pct"] = stats.get("weekly_diff_pct")

df = pd.DataFrame([row])
print(df.T)

                                       0
artist                            AVELLO
cm_id                            3623359
cm_artist_score                    70.57
sp_followers                       24374
sp_monthly_listeners             1461543
instagram_followers                77015
instagram_weekly_diff                476
instagram_weekly_diff_pct         0.6221
tiktok_followers                   33500
tiktok_weekly_diff                  None
tiktok_weekly_diff_pct              None
youtube_channel_followers            858
youtube_channel_weekly_diff         None
youtube_channel_weekly_diff_pct     None
twitter_followers                   None
twitter_weekly_diff                 None
twitter_weekly_diff_pct             None
soundcloud_followers               41080
soundcloud_weekly_diff               371
soundcloud_weekly_diff_pct        0.9113
deezer_followers                     114
deezer_weekly_diff                     1
deezer_weekly_diff_pct             0.885


In [10]:
playlists = cm.get_artist_playlists(3623359, platform="spotify", status="current", limit=20)

# Extract key playlist signals
editorial_playlists = [p for p in playlists if p['playlist'].get('editorial')]
total_playlist_reach = sum(p['playlist'].get('followers', 0) for p in playlists)
editorial_reach = sum(p['playlist'].get('followers', 0) for p in editorial_playlists)

playlist_summary = [{
    "playlist_name": p['playlist']['name'],
    "track": p['playlist']['track'],
    "followers": p['playlist']['followers'],
    "position": p['playlist']['position'],
    "peak_position": p['playlist']['peak_position'],
    "days_on_playlist": p['playlist']['period'],
    "editorial": p['playlist'].get('editorial'),
    "added_at": p['playlist']['added_at'],
} for p in playlists]

print(f"Total playlists: {len(playlists)}")
print(f"Editorial playlists: {len(editorial_playlists)}")
print(f"Total playlist reach: {total_playlist_reach:,}")
print(f"Editorial reach: {editorial_reach:,}")
print()
pd.DataFrame(playlist_summary)

Total playlists: 4
Editorial playlists: 4
Total playlist reach: 4,816,088
Editorial reach: 4,816,088



,playlist_name,track,followers,position,peak_position,days_on_playlist,editorial,added_at
0,Bass Arcade,Cry,991180,82,2,132,True,2025-10-10
1,Bass Arcade,Cry For You - AVELLO Remix,991180,25,8,28,True,2026-01-23
2,Bass Arcade,No Broke Boys - AVELLO Remix,991180,57,3,88,True,2025-11-24
3,Dance Rising,headrush,1842548,68,6,56,True,2025-12-26


In [11]:
row["spotify_playlist_count"] = len(playlists)
row["spotify_editorial_playlist_count"] = len(editorial_playlists)
row["spotify_total_playlist_reach"] = total_playlist_reach
row["spotify_editorial_reach"] = editorial_reach

df = pd.DataFrame([row])
print(df.T)

                                        0
artist                             AVELLO
cm_id                             3623359
cm_artist_score                     70.57
sp_followers                        24374
sp_monthly_listeners              1461543
instagram_followers                 77015
instagram_weekly_diff                 476
instagram_weekly_diff_pct          0.6221
tiktok_followers                    33500
tiktok_weekly_diff                   None
tiktok_weekly_diff_pct               None
youtube_channel_followers             858
youtube_channel_weekly_diff          None
youtube_channel_weekly_diff_pct      None
twitter_followers                    None
twitter_weekly_diff                  None
twitter_weekly_diff_pct              None
soundcloud_followers                41080
soundcloud_weekly_diff                371
soundcloud_weekly_diff_pct         0.9113
deezer_followers                      114
deezer_weekly_diff                      1
deezer_weekly_diff_pct            

In [14]:
am_playlists = cm.get_artist_playlists(3623359, platform="applemusic", status="current", limit=10)

# Get unique playlists and key signals
unique_playlists = {p['playlist']['id']: p['playlist'] for p in am_playlists}

am_playlist_count = len(unique_playlists)
am_total_countries = sum(p.get('num_countries', 0) for p in unique_playlists.values())
am_playlist_names = [p['name'] for p in unique_playlists.values()]

print(f"Apple Music playlists: {am_playlist_count}")
print(f"Total country placements: {am_total_countries}")
print(f"Playlists: {am_playlist_names}")

# Add to row
row["applemusic_playlist_count"] = am_playlist_count
row["applemusic_total_country_placements"] = am_total_countries

df = pd.DataFrame([row])
print(df.T)

Apple Music playlists: 5
Total country placements: 640
Playlists: ['Melodic Bass', 'Breaking Dance', 'New in Dance', 'Trap & Bass', 'Heavy Hitters']
                                           0
artist                                AVELLO
cm_id                                3623359
cm_artist_score                        70.57
sp_followers                           24374
sp_monthly_listeners                 1461543
instagram_followers                    77015
instagram_weekly_diff                    476
instagram_weekly_diff_pct             0.6221
tiktok_followers                       33500
tiktok_weekly_diff                      None
tiktok_weekly_diff_pct                  None
youtube_channel_followers                858
youtube_channel_weekly_diff             None
youtube_channel_weekly_diff_pct         None
twitter_followers                       None
twitter_weekly_diff                     None
twitter_weekly_diff_pct                 None
soundcloud_followers                   41

In [15]:
combined_audience = (
    (row.get("sp_followers") or 0) +
    (row.get("instagram_followers") or 0) +
    (row.get("youtube_channel_followers") or 0) +
    (row.get("tiktok_followers") or 0)
)

row["combined_audience_size"] = combined_audience

print(f"Combined Audience Size: {combined_audience:,}")
print(f"  Spotify:   {row.get('sp_followers'):,}")
print(f"  Instagram: {row.get('instagram_followers'):,}")
print(f"  YouTube:   {row.get('youtube_channel_followers'):,}")
print(f"  TikTok:    {row.get('tiktok_followers'):,}")

Combined Audience Size: 135,747
  Spotify:   24,374
  Instagram: 77,015
  YouTube:   858
  TikTok:    33,500


In [16]:
for platform, key in [("Spotify", "sp_followers"), ("Instagram", "instagram_followers"), 
                       ("YouTube", "youtube_channel_followers"), ("TikTok", "tiktok_followers")]:
    val = row.get(key) or 0
    pct = (val / combined_audience * 100) if combined_audience else 0
    print(f"  {platform}: {pct:.1f}%")

  Spotify: 18.0%
  Instagram: 56.7%
  YouTube: 0.6%
  TikTok: 24.7%


In [17]:
row["combined_audience_size"] = combined_audience
row["sp_audience_pct"] = round((row.get("sp_followers") or 0) / combined_audience * 100, 1)
row["instagram_audience_pct"] = round((row.get("instagram_followers") or 0) / combined_audience * 100, 1)
row["youtube_audience_pct"] = round((row.get("youtube_channel_followers") or 0) / combined_audience * 100, 1)
row["tiktok_audience_pct"] = round((row.get("tiktok_followers") or 0) / combined_audience * 100, 1)

df = pd.DataFrame([row])
print(df.T)

                                           0
artist                                AVELLO
cm_id                                3623359
cm_artist_score                        70.57
sp_followers                           24374
sp_monthly_listeners                 1461543
instagram_followers                    77015
instagram_weekly_diff                    476
instagram_weekly_diff_pct             0.6221
tiktok_followers                       33500
tiktok_weekly_diff                      None
tiktok_weekly_diff_pct                  None
youtube_channel_followers                858
youtube_channel_weekly_diff             None
youtube_channel_weekly_diff_pct         None
twitter_followers                       None
twitter_weekly_diff                     None
twitter_weekly_diff_pct                 None
soundcloud_followers                   41080
soundcloud_weekly_diff                   371
soundcloud_weekly_diff_pct            0.9113
deezer_followers                         114
deezer_wee